# DỰ ÁN CUỐI KHÓA "THE PRICE IS RIGHT"

## Tóm tắt quy trình của notebook

Notebook thực hiện một quy trình dự đoán giá sản phẩm từ dữ liệu Amazon. Quy trình bắt đầu bằng việc import thư viện và tải dữ liệu, sau đó xây dựng các mô hình cơ sở để có mốc so sánh. Tiếp theo, notebook tạo các đặc trưng đơn giản như cân nặng và độ dài mô tả, huấn luyện hồi quy tuyến tính, chuyển văn bản thành vector bằng `CountVectorizer`, rồi thử nghiệm Linear Regression, Random Forest và XGBoost. Cuối cùng, các mô hình được đánh giá trên tập kiểm tra để so sánh chất lượng dự đoán.

## Ý nghĩa chính của notebook

Notebook giải quyết bài toán ước lượng giá của sản phẩm dựa trên thông tin mô tả và một số đặc trưng liên quan. Dữ liệu được chia thành các tập huấn luyện, kiểm định và kiểm tra; mô hình chỉ học từ dữ liệu huấn luyện, sau đó dự đoán trên dữ liệu chưa thấy để đánh giá khả năng tổng quát hóa. Các kết quả đánh giá giúp người học hiểu mức độ hiệu quả của từng cách tiếp cận và chọn hướng phát triển tốt hơn cho dự án.

# Trình tự thực hiện

NGÀY 1: Tuyển chọn dữ liệu  
NGÀY 2: Tiền xử lý dữ liệu  
NGÀY 3: Đánh giá, mô hình cơ sở và học máy truyền thống  
NGÀY 4: Học sâu và LLM  
NGÀY 5: Tinh chỉnh một mô hình nền tảng  

## NGÀY 3: Đánh giá, mô hình cơ sở và học máy truyền thống

Hôm nay, chúng ta sẽ viết một số mô hình đơn giản để dự đoán giá sản phẩm.

Chúng ta sẽ sử dụng một phương pháp để đánh giá hiệu quả của mô hình.

Sau đó, chúng ta sẽ thử nghiệm các mô hình cơ sở bằng học máy truyền thống.

## Mục tiêu cuối cùng

Sau khi chạy và hiểu notebook, người học có thể xây dựng một quy trình dự đoán giá hoàn chỉnh, biết cách tạo đặc trưng từ dữ liệu dạng bảng và văn bản, huấn luyện nhiều mô hình học máy truyền thống, đồng thời so sánh kết quả để chọn mô hình phù hợp cho bài toán thực tế.

## Giải thích từng cell

Các phần giải thích chi tiết của từng cell được trình bày ngay sau cell tương ứng. Mỗi phần nêu mục đích, hoạt động chính, vai trò trong quy trình và ý nghĩa thực tế của cell.

In [ ]:
import random
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
from pricer.evaluator import evaluate
from pricer.items import Item

### Giải thích Cell 2

- **Mục đích:** Chuẩn bị các thư viện cần thiết cho toàn bộ quy trình dự đoán giá.
- **Lệnh chính:** `random` tạo số ngẫu nhiên; `pandas` và `numpy` xử lý dữ liệu; các lớp từ `sklearn` xây dựng mô hình, tính chỉ số đánh giá và biến đổi văn bản; `evaluate` đánh giá mô hình; `Item` tải và biểu diễn dữ liệu sản phẩm.
- **Vai trò trong quy trình:** Các thư viện này phải được nạp trước khi những cell phía sau gọi đến mô hình, dữ liệu hoặc hàm đánh giá.
- **Kết quả sử dụng tiếp theo:** Tên thư viện và lớp đã import được dùng trong các cell tải dữ liệu, huấn luyện và đánh giá mô hình.

**Ý nghĩa thực tế:** Cell này tạo nền tảng công cụ cho dự án, giúp các bước xử lý dữ liệu và học máy phía sau có thể chạy thống nhất.

In [ ]:
LITE_MODE = False

### Giải thích Cell 3

- **Mục đích:** Chọn chế độ dữ liệu dùng trong notebook.
- **Lệnh chính:** `LITE_MODE = False` yêu cầu sử dụng tập dữ liệu đầy đủ thay vì tập rút gọn.
- **Vai trò trong quy trình:** Biến này điều khiển tên dataset ở Cell 4, giúp người học có thể chuyển nhanh giữa chạy thử nhẹ và chạy đầy đủ.
- **Kết quả sử dụng tiếp theo:** Giá trị `LITE_MODE` được dùng để tạo đường dẫn dataset.

**Ý nghĩa thực tế:** Cell này cho phép cân bằng giữa tốc độ thử nghiệm và độ đầy đủ của dữ liệu khi phát triển mô hình.

In [ ]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Đã tải {len(train):,} mẫu huấn luyện, {len(val):,} mẫu kiểm định và {len(test):,} mẫu kiểm tra")

### Giải thích Cell 4

- **Mục đích:** Xác định dataset và tải dữ liệu sản phẩm.
- **Lệnh chính:** `dataset` chọn phiên bản `items_lite` hoặc `items_full`; `Item.from_hub(dataset)` tải dữ liệu và chia thành `train`, `val`, `test`; `print()` hiển thị số lượng mẫu của từng tập.
- **Vai trò trong quy trình:** Mô hình cần dữ liệu huấn luyện để học, dữ liệu kiểm định để theo dõi và dữ liệu kiểm tra để đánh giá cuối cùng.
- **Kết quả sử dụng tiếp theo:** Các biến `train`, `val` và `test` được dùng bởi những mô hình cơ sở và mô hình học máy sau đó.

**Ý nghĩa thực tế:** Cell này đưa dữ liệu sản phẩm vào quy trình và kiểm tra nhanh quy mô dữ liệu trước khi huấn luyện.

In [ ]:
def random_pricer(item):
    return random.randrange(1,1000)

### Giải thích Cell 5

- **Mục đích:** Tạo mô hình cơ sở ngẫu nhiên để làm mốc so sánh.
- **Lệnh chính:** Hàm `random_pricer(item)` nhận một sản phẩm nhưng trả về một số nguyên ngẫu nhiên từ 1 đến 999 bằng `random.randrange(1, 1000)`.
- **Vai trò trong quy trình:** Một mô hình đơn giản, gần như không học từ dữ liệu, giúp xác định mức chất lượng tối thiểu mà các mô hình sau cần vượt qua.
- **Kết quả sử dụng tiếp theo:** Hàm `random_pricer` được truyền vào `evaluate` ở Cell 6.

**Ý nghĩa thực tế:** Cell này tạo đường chuẩn thấp để biết một cách dự đoán ngẫu nhiên hoạt động kém đến mức nào.

In [ ]:
random.seed(42)
evaluate(random_pricer, test)

### Giải thích Cell 6

- **Mục đích:** Đánh giá mô hình ngẫu nhiên trên tập kiểm tra.
- **Lệnh chính:** `random.seed(42)` cố định trạng thái ngẫu nhiên để kết quả có thể lặp lại; `evaluate(random_pricer, test)` chạy hàm dự đoán trên các sản phẩm trong `test` và tính chỉ số đánh giá.
- **Vai trò trong quy trình:** Đây là kết quả mốc ban đầu để so sánh với mô hình dự đoán giá trung bình và các mô hình học máy.
- **Kết quả sử dụng tiếp theo:** Kết quả đánh giá được dùng để nhận xét liệu các mô hình tiếp theo có cải thiện hay không.

**Ý nghĩa thực tế:** Cell này đo chất lượng tối thiểu của hệ thống trước khi đầu tư vào các mô hình có khả năng học từ dữ liệu.

In [ ]:
# Khá thú vị!
# Chúng ta có thể làm tốt hơn với một mô hình đơn giản khác

training_prices = [item.price for item in train]
training_average = sum(training_prices) / len(training_prices)
print(f"Giá trung bình trong tập huấn luyện: {training_average}")

def constant_pricer(item):
    return training_average

### Giải thích Cell 7

- **Mục đích:** Tạo mô hình cơ sở dự đoán cùng một mức giá cho mọi sản phẩm.
- **Lệnh chính:** Danh sách `training_prices` lấy giá từ dữ liệu huấn luyện; `training_average` tính giá trung bình; `constant_pricer(item)` trả về giá trung bình đó cho mọi sản phẩm.
- **Vai trò trong quy trình:** Mô hình này đơn giản hơn mô hình học máy nhưng thường ổn định hơn dự đoán ngẫu nhiên.
- **Kết quả sử dụng tiếp theo:** Hàm `constant_pricer` được đánh giá ở Cell 8.

**Ý nghĩa thực tế:** Cell này cho biết chỉ riêng việc dùng giá trung bình đã có thể tạo ra một mốc dự đoán hữu ích như thế nào.

In [ ]:
evaluate(constant_pricer, test)

### Giải thích Cell 8

- **Mục đích:** Đo chất lượng của mô hình giá trung bình trên tập kiểm tra.
- **Lệnh chính:** `evaluate(constant_pricer, test)` gọi hàm dự đoán cho từng sản phẩm trong `test` và tổng hợp sai số.
- **Vai trò trong quy trình:** Kết quả này là mốc cơ sở thứ hai, thường có ý nghĩa hơn mốc ngẫu nhiên vì được tính từ dữ liệu huấn luyện.
- **Kết quả sử dụng tiếp theo:** Kết quả được dùng để so sánh với hồi quy tuyến tính và các mô hình dựa trên văn bản.

**Ý nghĩa thực tế:** Cell này kiểm tra liệu một chiến lược đơn giản dựa trên thống kê tổng quát có đủ tốt cho bài toán hay không.

In [ ]:
def get_features(item):
    return {
        "weight": item.weight,
        "weight_unknown": 1 if item.weight==0 else 0,
        "text_length": len(item.summary)
    }

### Giải thích Cell 9

- **Mục đích:** Chuyển mỗi sản phẩm thành một tập đặc trưng dạng số.
- **Lệnh chính:** `item.weight` lấy cân nặng; `weight_unknown` đánh dấu trường hợp thiếu cân nặng bằng giá trị 1; `len(item.summary)` đo độ dài phần mô tả.
- **Vai trò trong quy trình:** Các mô hình học máy truyền thống cần đầu vào dạng số, vì vậy thông tin sản phẩm phải được biểu diễn thành các đặc trưng có thể tính toán.
- **Kết quả sử dụng tiếp theo:** Dictionary đặc trưng được dùng để tạo DataFrame ở Cell 10 và dự đoán một sản phẩm mới ở Cell 12.

**Ý nghĩa thực tế:** Cell này biến thông tin sản phẩm thành các tín hiệu định lượng để mô hình có thể học mối liên hệ giữa đặc trưng và giá.

In [ ]:
def list_to_dataframe(items):
    features = [get_features(item) for item in items]
    df = pd.DataFrame(features)
    df['price'] = [item.price for item in items]
    return df

train_df = list_to_dataframe(train)
test_df = list_to_dataframe(test)

### Giải thích Cell 10

- **Mục đích:** Tạo bảng dữ liệu đặc trưng và thêm giá mục tiêu cho nhiều sản phẩm.
- **Lệnh chính:** List comprehension gọi `get_features` cho từng item; `pd.DataFrame(features)` tạo bảng; cột `price` chứa giá thật của từng sản phẩm.
- **Vai trò trong quy trình:** DataFrame giúp tách rõ biến đầu vào và biến mục tiêu, phù hợp với API của scikit-learn.
- **Kết quả sử dụng tiếp theo:** `train_df` và `test_df` được dùng để chuẩn bị dữ liệu cho hồi quy tuyến tính ở Cell 11.

**Ý nghĩa thực tế:** Cell này tạo bảng học máy chuẩn, trong đó mỗi dòng là một sản phẩm và mỗi cột mô tả một đặc trưng hoặc giá cần dự đoán.

In [ ]:
# Hồi quy tuyến tính truyền thống!

np.random.seed(42)

# Tách các đặc trưng và biến mục tiêu
feature_columns = ['weight', 'weight_unknown', 'text_length']

X_train = train_df[feature_columns]
y_train = train_df['price']
X_test = test_df[feature_columns]
y_test = test_df['price']

# Huấn luyện mô hình hồi quy tuyến tính
model = LinearRegression()
model.fit(X_train, y_train)

for feature, coef in zip(feature_columns, model.coef_):
    print(f"{feature}: {coef}")
print(f"Hệ số chặn: {model.intercept_}")

# Dự đoán trên tập kiểm tra và đánh giá
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Sai số bình phương trung bình: {mse}")
print(f"Điểm R bình phương: {r2}")

### Giải thích Cell 11

- **Mục đích:** Huấn luyện và đánh giá mô hình hồi quy tuyến tính trên các đặc trưng cơ bản.
- **Lệnh chính:** `feature_columns` chọn ba cột đầu vào; `model.fit` học hệ số; vòng lặp in mức ảnh hưởng của từng đặc trưng; `model.predict` tạo dự đoán; `mean_squared_error` và `r2_score` đo sai số và mức độ giải thích của mô hình.
- **Vai trò trong quy trình:** Đây là mô hình học máy truyền thống đầu tiên sử dụng đặc trưng có ý nghĩa thay vì dự đoán ngẫu nhiên hoặc một giá trị cố định.
- **Kết quả sử dụng tiếp theo:** Mô hình `model` và danh sách `feature_columns` được dùng để xây dựng hàm dự đoán ở Cell 12.

**Ý nghĩa thực tế:** Cell này cho thấy cân nặng và độ dài mô tả ảnh hưởng thế nào đến giá, đồng thời cung cấp một mô hình cơ sở có thể diễn giải được.

In [ ]:
def linear_regression_pricer(item):
    features = get_features(item)
    features_df = pd.DataFrame([features])
    return model.predict(features_df)[0]

### Giải thích Cell 12

- **Mục đích:** Đóng gói quy trình tạo đặc trưng và dự đoán thành một hàm có thể dùng cho từng sản phẩm.
- **Lệnh chính:** `get_features(item)` tạo dictionary đặc trưng; `pd.DataFrame([features])` đưa một sản phẩm về đúng dạng bảng; `model.predict` trả về giá dự đoán.
- **Vai trò trong quy trình:** Hàm này nối mô hình đã huấn luyện với giao diện dự đoán từng item mà hàm đánh giá yêu cầu.
- **Kết quả sử dụng tiếp theo:** `linear_regression_pricer` được đánh giá trên toàn bộ tập test ở Cell 13.

**Ý nghĩa thực tế:** Cell này biến mô hình hồi quy thành một hàm dự đoán có thể tái sử dụng trong ứng dụng giá sản phẩm.

In [ ]:
evaluate(linear_regression_pricer, test)

### Giải thích Cell 13

- **Mục đích:** Đánh giá hàm dự đoán của hồi quy tuyến tính trên dữ liệu chưa dùng để kiểm tra trong quá trình huấn luyện.
- **Lệnh chính:** `evaluate(linear_regression_pricer, test)` chạy hàm dự đoán trên từng sản phẩm trong `test` và tổng hợp chất lượng.
- **Vai trò trong quy trình:** Việc đánh giá trên tập test cho biết mô hình có tổng quát hóa được hay chỉ phù hợp với dữ liệu huấn luyện.
- **Kết quả sử dụng tiếp theo:** Kết quả là mốc so sánh với các mô hình dùng toàn bộ nội dung mô tả.

**Ý nghĩa thực tế:** Cell này kiểm chứng mô hình đặc trưng cơ bản trước khi mở rộng đầu vào bằng thông tin ngôn ngữ tự nhiên.

In [ ]:
prices = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

### Giải thích Cell 14

- **Mục đích:** Chuẩn bị giá mục tiêu và nội dung mô tả sản phẩm cho mô hình dựa trên văn bản.
- **Lệnh chính:** List comprehension lấy giá và chuyển thành mảng số bằng `np.array`; danh sách `documents` chứa phần tóm tắt của từng sản phẩm.
- **Vai trò trong quy trình:** Mô hình ngôn ngữ cần văn bản đầu vào, còn thuật toán hồi quy cần nhãn giá dạng số.
- **Kết quả sử dụng tiếp theo:** `prices` và `documents` được dùng để huấn luyện vectorizer và regressor ở các cell sau.

**Ý nghĩa thực tế:** Cell này tách dữ liệu thành văn bản và giá thật, tạo đầu vào cho hướng dự đoán giá từ nội dung mô tả.

In [ ]:
np.random.seed(42)
vectorizer = CountVectorizer(max_features=2000, stop_words='english')
X = vectorizer.fit_transform(documents)


### Giải thích Cell 15

- **Mục đích:** Biến mô tả sản phẩm thành ma trận số dựa trên số lần xuất hiện của từ.
- **Lệnh chính:** `CountVectorizer` chọn tối đa 2.000 từ và bỏ qua stop words tiếng Anh; `fit_transform(documents)` học từ vựng trên dữ liệu huấn luyện rồi tạo vector cho từng mô tả.
- **Vai trò trong quy trình:** Mô hình hồi quy không xử lý trực tiếp chuỗi văn bản, nên văn bản cần được mã hóa thành các con số.
- **Kết quả sử dụng tiếp theo:** Ma trận `X` được dùng để xem từ được chọn và huấn luyện Linear Regression, Random Forest và XGBoost.

**Ý nghĩa thực tế:** Cell này chuyển ngôn ngữ tự nhiên của mô tả sản phẩm thành dữ liệu số để thuật toán học máy có thể phân tích.

In [ ]:
# Đây là 1.000 từ phổ biến nhất được chọn, không bao gồm "stop words":

selected_words = vectorizer.get_feature_names_out()
print(f"Số lượng từ được chọn: {len(selected_words)}")
print("Các từ được chọn:", selected_words[1000:1020])

### Giải thích Cell 16

- **Mục đích:** Kiểm tra số lượng và một phần danh sách từ mà `CountVectorizer` đã chọn.
- **Lệnh chính:** `get_feature_names_out()` lấy tên các từ trong từ vựng; `len` đếm số từ; slicing `[1000:1020]` hiển thị một đoạn danh sách để quan sát.
- **Vai trò trong quy trình:** Kiểm tra từ vựng giúp người học hiểu văn bản đã được biểu diễn như thế nào và phát hiện cấu hình vectorizer không phù hợp.
- **Kết quả sử dụng tiếp theo:** Các từ được chọn chính là các cột của ma trận `X` dùng trong các mô hình phía sau.

**Ý nghĩa thực tế:** Cell này giúp kiểm tra chất lượng bước tiền xử lý văn bản trước khi dùng vector để dự đoán giá.

In [ ]:
regressor = LinearRegression()
regressor.fit(X, prices)

### Giải thích Cell 17

- **Mục đích:** Huấn luyện hồi quy tuyến tính bằng các vector từ và giá sản phẩm.
- **Lệnh chính:** `LinearRegression()` tạo mô hình; `regressor.fit(X, prices)` học mối quan hệ giữa số lần xuất hiện của các từ và giá.
- **Vai trò trong quy trình:** Đây là mô hình đầu tiên sử dụng trực tiếp nội dung mô tả thay vì chỉ dùng cân nặng và độ dài văn bản.
- **Kết quả sử dụng tiếp theo:** Mô hình `regressor` được gọi trong hàm dự đoán ở Cell 18.

**Ý nghĩa thực tế:** Cell này kiểm tra xem các từ xuất hiện trong mô tả có thể cung cấp tín hiệu hữu ích để ước lượng giá hay không.

In [ ]:
def natural_language_linear_regression_pricer(item):
    x = vectorizer.transform([item.summary])
    return max(regressor.predict(x)[0], 0)

### Giải thích Cell 18

- **Mục đích:** Tạo hàm dự đoán giá từ nội dung mô tả của một sản phẩm.
- **Lệnh chính:** `vectorizer.transform` dùng đúng từ vựng đã học để biến mô tả mới thành vector; `regressor.predict` dự đoán giá; `max(..., 0)` ngăn kết quả âm.
- **Vai trò trong quy trình:** Hàm này bảo đảm sản phẩm mới được xử lý giống dữ liệu huấn luyện trước khi dự đoán.
- **Kết quả sử dụng tiếp theo:** `natural_language_linear_regression_pricer` được đánh giá trên tập test ở Cell 19.

**Ý nghĩa thực tế:** Cell này mô phỏng cách hệ thống nhận một mô tả sản phẩm mới và trả về mức giá ước lượng.

In [ ]:
evaluate(natural_language_linear_regression_pricer, test)

### Giải thích Cell 19

- **Mục đích:** Đánh giá mô hình hồi quy dựa trên nội dung mô tả.
- **Lệnh chính:** `evaluate(natural_language_linear_regression_pricer, test)` chạy pipeline vector hóa và dự đoán cho toàn bộ tập test.
- **Vai trò trong quy trình:** Kết quả cho biết việc dùng từ trong mô tả có cải thiện so với đặc trưng cơ bản và giá trung bình hay không.
- **Kết quả sử dụng tiếp theo:** Kết quả được dùng làm mốc so sánh với Random Forest và XGBoost.

**Ý nghĩa thực tế:** Cell này đo trực tiếp giá trị của thông tin văn bản trong bài toán định giá sản phẩm.

In [ ]:
subset = 15_000
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
rf_model.fit(X[:subset], prices[:subset])

### Giải thích Cell 20

- **Mục đích:** Huấn luyện Random Forest trên một phần dữ liệu vector hóa.
- **Lệnh chính:** `subset = 15_000` giới hạn số mẫu; `RandomForestRegressor` tạo 100 cây quyết định; `fit` huấn luyện mô hình trên các vector và giá tương ứng.
- **Vai trò trong quy trình:** Random Forest có thể học các quan hệ phi tuyến và tương tác giữa nhiều từ, khác với hồi quy tuyến tính.
- **Kết quả sử dụng tiếp theo:** `rf_model` được dùng để dự đoán ở Cell 22 sau phần giải thích về thuật toán.

**Ý nghĩa thực tế:** Cell này thử một mô hình ensemble mạnh hơn trong giới hạn thời gian và tài nguyên tính toán phù hợp cho việc học tập.

## Mô hình Random Forest

Random Forest là một dạng thuật toán **ensemble**, nghĩa là nó kết hợp nhiều thuật toán nhỏ hơn để đưa ra dự đoán tốt hơn.

Nó sử dụng một loại thuật toán học máy rất đơn giản gọi là **cây quyết định**. Cây quyết định đưa ra dự đoán bằng cách kiểm tra các giá trị đặc trưng đầu vào, tương tự một lưu đồ gồm các câu lệnh IF. Cây quyết định nhanh và đơn giản nhưng thường có xu hướng overfit.

Trong trường hợp này, "đặc trưng" là các phần tử của vector, nói cách khác là số lần một từ cụ thể xuất hiện trong phần mô tả sản phẩm.

Có thể hình dung như sau:

**Cây quyết định**  
\- NẾU từ "TV" xuất hiện nhiều hơn 3 lần THÌ  
-- NẾU từ "LED" xuất hiện nhiều hơn 2 lần THÌ  
--- NẾU từ "HD" xuất hiện ít nhất một lần THÌ  
---- Giá = 500 USD


Random Forest tạo ra nhiều cây quyết định. Mỗi cây được huấn luyện trên một tập con ngẫu nhiên khác nhau của dữ liệu và một tập con ngẫu nhiên khác nhau của các đặc trưng. Ở trên, chúng ta chỉ định 100 cây, đây cũng là giá trị mặc định.

Sau đó, mô hình Random Forest lấy giá trị trung bình dự đoán từ tất cả các cây để tạo ra kết quả cuối cùng.

### Giải thích Cell 21

- **Mục đích:** Giải thích nguyên lý của Random Forest trước khi sử dụng nó để dự đoán.
- **Nội dung chính:** Cell mô tả cây quyết định, cách nhiều cây được huấn luyện trên các mẫu và đặc trưng khác nhau, rồi lấy trung bình dự đoán.
- **Vai trò trong quy trình:** Phần giải thích giúp liên kết ma trận vector từ với cách Random Forest phân chia dữ liệu và học quan hệ phi tuyến.
- **Kết quả sử dụng tiếp theo:** Người học dùng kiến thức này để hiểu hàm `random_forest` và kết quả đánh giá ở các cell sau.

**Ý nghĩa thực tế:** Cell này giúp biến một thuật toán ensemble khó hình dung thành quy trình gần gũi, từ đó dễ đánh giá ưu nhược điểm khi áp dụng vào định giá.

In [ ]:
def random_forest(item):
    x = vectorizer.transform([item.summary])
    return max(0, rf_model.predict(x)[0])

### Giải thích Cell 22

- **Mục đích:** Tạo hàm dự đoán giá bằng Random Forest.
- **Lệnh chính:** Mô tả sản phẩm được biến đổi bằng `vectorizer.transform`; `rf_model.predict` tạo dự đoán; `max(0, ...)` bảo đảm giá không âm.
- **Vai trò trong quy trình:** Hàm này đưa Random Forest vào cùng giao diện dự đoán với các mô hình trước, nên có thể đánh giá bằng cùng một hàm `evaluate`.
- **Kết quả sử dụng tiếp theo:** `random_forest` được đánh giá ở Cell 23.

**Ý nghĩa thực tế:** Cell này biến mô hình Random Forest đã huấn luyện thành chức năng định giá có thể gọi cho từng sản phẩm.

In [ ]:
evaluate(random_forest, test)

### Giải thích Cell 23

- **Mục đích:** Đo chất lượng dự đoán của Random Forest.
- **Lệnh chính:** `evaluate(random_forest, test)` chạy hàm Random Forest trên tập kiểm tra và tính các chỉ số đánh giá.
- **Vai trò trong quy trình:** Kết quả cho phép so sánh mô hình ensemble với hồi quy tuyến tính dùng cùng biểu diễn văn bản.
- **Kết quả sử dụng tiếp theo:** Kết quả là cơ sở để cân nhắc chi phí tính toán và chất lượng trước khi thử XGBoost.

**Ý nghĩa thực tế:** Cell này cho biết việc kết hợp nhiều cây quyết định có giúp hệ thống định giá chính xác hơn hay không.

In [ ]:
# Đây là cách lưu mô hình nếu bạn cần, đặc biệt khi chạy trên một tập dữ liệu lớn hơn

# import joblib
# joblib.dump(rf_model, "random_forest.joblib")

### Giải thích Cell 24

- **Mục đích:** Minh họa cách lưu mô hình Random Forest để dùng lại sau này.
- **Lệnh chính:** Hai dòng `joblib` đang được comment; nếu bỏ comment, `joblib.dump` sẽ ghi mô hình vào file `random_forest.joblib`.
- **Vai trò trong quy trình:** Lưu mô hình giúp tránh phải huấn luyện lại từ đầu khi triển khai hoặc đánh giá sau này.
- **Kết quả sử dụng tiếp theo:** File mô hình có thể được nạp lại trong một chương trình dự đoán riêng.

**Ý nghĩa thực tế:** Cell này minh họa bước chuyển từ thử nghiệm trong notebook sang lưu trữ mô hình để tái sử dụng trong ứng dụng.

## Giới thiệu XGBoost

Giống Random Forest, XGBoost cũng là một mô hình ensemble kết hợp nhiều cây quyết định.

Tuy nhiên, khác với Random Forest, XGBoost xây dựng từng cây nối tiếp nhau; mỗi cây tiếp theo sẽ sửa các lỗi của cây trước đó bằng phương pháp "gradient descent".

XGBoost nhanh hơn nhiều so với Random Forest, vì vậy chúng ta có thể chạy trên toàn bộ tập dữ liệu. Mô hình này thường có khả năng tổng quát hóa tốt hơn.

**Nếu lệnh import này không hoạt động, bạn có thể bỏ qua phần này vì đây không phải nội dung bắt buộc. Trên máy Mac, có thể cần chạy `brew install libomp` trong terminal.**

### Giải thích Cell 25

- **Mục đích:** Giới thiệu hướng tiếp cận XGBoost và điều kiện môi trường có thể ảnh hưởng đến việc import thư viện.
- **Nội dung chính:** Cell so sánh XGBoost với Random Forest, giải thích việc xây dựng cây nối tiếp và nhắc rằng một số hệ thống có thể cần thư viện hệ điều hành bổ sung.
- **Vai trò trong quy trình:** Phần này chuẩn bị kiến thức trước khi tạo và huấn luyện mô hình XGBoost ở các cell sau.
- **Kết quả sử dụng tiếp theo:** Người học biết khi nào có thể bỏ qua phần XGBoost nếu môi trường chưa cài đủ phụ thuộc.

**Ý nghĩa thực tế:** Cell này đặt mô hình XGBoost trong bối cảnh triển khai thực tế, nơi hiệu năng và cấu hình môi trường đều cần được cân nhắc.

In [ ]:
import xgboost as xgb

### Giải thích Cell 26

- **Mục đích:** Import thư viện XGBoost để sử dụng mô hình hồi quy dạng gradient boosting.
- **Lệnh chính:** `import xgboost as xgb` tạo bí danh ngắn `xgb` để gọi các lớp và hàm của thư viện.
- **Vai trò trong quy trình:** Cell này cung cấp công cụ cho mô hình cuối cùng của notebook.
- **Kết quả sử dụng tiếp theo:** `xgb.XGBRegressor` được dùng để tạo mô hình ở Cell 27.

**Ý nghĩa thực tế:** Cell này kết nối notebook với một thư viện học máy chuyên dụng để thử nghiệm mô hình mạnh hơn.

In [ ]:
np.random.seed(42)

xgb_model = xgb.XGBRegressor(n_estimators=1000, random_state=42, n_jobs=4, learning_rate=0.1)
xgb_model.fit(X, prices)

### Giải thích Cell 27

- **Mục đích:** Tạo và huấn luyện mô hình XGBoost trên toàn bộ dữ liệu vector hóa.
- **Lệnh chính:** `XGBRegressor` cấu hình 1.000 cây, tốc độ học `0.1`, trạng thái ngẫu nhiên cố định và số luồng xử lý; `fit(X, prices)` học từ vector từ và giá.
- **Vai trò trong quy trình:** XGBoost xây dựng các cây nối tiếp để sửa lỗi của các cây trước, cung cấp một hướng tiếp cận mạnh cho dữ liệu có quan hệ phức tạp.
- **Kết quả sử dụng tiếp theo:** Mô hình `xgb_model` được gọi trong hàm dự đoán ở Cell 28.

**Ý nghĩa thực tế:** Cell này huấn luyện mô hình có khả năng khai thác quan hệ phi tuyến giữa nội dung mô tả và giá sản phẩm.

In [ ]:
def xg_boost(item):
    x = vectorizer.transform([item.summary])
    return max(0, xgb_model.predict(x)[0])

### Giải thích Cell 28

- **Mục đích:** Tạo hàm dự đoán giá bằng mô hình XGBoost.
- **Lệnh chính:** `vectorizer.transform` biến mô tả mới thành vector theo đúng từ vựng đã học; `xgb_model.predict` tạo giá dự đoán; `max(0, ...)` loại bỏ giá trị âm.
- **Vai trò trong quy trình:** Hàm này chuẩn hóa cách gọi XGBoost theo cùng giao diện với các mô hình trước.
- **Kết quả sử dụng tiếp theo:** `xg_boost` được đánh giá ở Cell 29.

**Ý nghĩa thực tế:** Cell này đóng gói mô hình XGBoost thành chức năng có thể phục vụ yêu cầu định giá sản phẩm mới.

In [ ]:
evaluate(xg_boost, test)

### Giải thích Cell 29

- **Mục đích:** Đánh giá kết quả cuối của XGBoost trên tập kiểm tra.
- **Lệnh chính:** `evaluate(xg_boost, test)` chạy quy trình vector hóa và dự đoán cho từng sản phẩm, sau đó tính chỉ số đánh giá.
- **Vai trò trong quy trình:** Đây là phép đo cuối để so sánh XGBoost với mô hình ngẫu nhiên, giá trung bình, hồi quy tuyến tính và Random Forest.
- **Kết quả sử dụng tiếp theo:** Kết quả giúp chọn mô hình phù hợp dựa trên độ chính xác, thời gian huấn luyện và tài nguyên.

**Ý nghĩa thực tế:** Cell này cung cấp bằng chứng định lượng để quyết định mô hình nào đáng được đưa vào bước phát triển tiếp theo.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Ứng dụng trong kinh doanh</h2>
            <span style="color:#181;">Học máy truyền thống không chỉ hữu ích để tìm hiểu lịch sử; ngày nay nó vẫn được sử dụng rộng rãi trong ngành, đặc biệt với các bài toán có những đặc trưng nhận diện rõ ràng. Hãy dành thời gian khám phá các thuật toán và thử nghiệm. Biết đâu bạn có thể vượt qua các kết quả của tôi bằng học máy truyền thống! Tôi đã chạy Random Forest trên toàn bộ 800.000 mẫu huấn luyện. Quá trình này mất khoảng 15 giờ và đạt sai số thấp ở mức 56,40 USD. Học máy truyền thống có thể cho kết quả tốt, hãy tự mình thử nghiệm.</span>
        </td>
    </tr>
</table>

### Giải thích Cell 30

- **Mục đích:** Liên hệ kết quả học máy với khả năng ứng dụng trong kinh doanh.
- **Nội dung chính:** Cell nhấn mạnh học máy truyền thống vẫn hữu ích khi dữ liệu có đặc trưng rõ ràng và đưa ra ví dụ về thời gian chạy cùng sai số của Random Forest trên tập dữ liệu lớn.
- **Vai trò trong quy trình:** Phần này giúp chuyển từ kết quả thử nghiệm sang góc nhìn lựa chọn mô hình và triển khai trong thực tế.
- **Kết quả sử dụng tiếp theo:** Người học có thể dùng các kết quả benchmark làm mốc để tiếp tục cải thiện mô hình.

**Ý nghĩa thực tế:** Cell này cho thấy một mô hình có giá trị không chỉ vì thuật toán, mà còn vì mức độ phù hợp giữa độ chính xác, thời gian chạy và nhu cầu kinh doanh.